# Ablation 3: Reranker — Người 3 (Kiệt)

| # | Config | Base Retrieval | Fusion | Reranker | Mục tiêu |
|---|--------|----------------|--------|----------|----------|
| 1 | `Rerank-None-Hybrid` | Dense + BM25 | Score merge | ❌ | Hybrid baseline (no reranking) |
| 2 | `Rerank-RRF-Hybrid` | Dense + BM25 | RRF | ❌ | RRF fusion only |
| 3 | `Rerank-CrossEncoder-Hybrid` | Dense + BM25 | Score merge | CrossEncoder | CE rerank on merged |
| 4 | `Rerank-RRF+CrossEncoder-Hybrid` | Dense + BM25 | RRF | CrossEncoder | RRF → CE rerank |

**GPU**: Auto-detect T4, dùng GPU cho embedding + Cross-Encoder  
**Cache**: Dense + BM25 lưu pickle, lần sau load lại không cần chạy lại  

---

## 1. Install & Imports

In [1]:
# faiss-gpu cho Kaggle T4, fallback faiss-cpu
import subprocess, sys
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu'], 
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('✅ faiss-gpu installed')
except:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('⚠️ faiss-gpu failed, using faiss-cpu')
!pip install -q rank_bm25 sentence-transformers underthesea openai

✅ faiss-gpu installed
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 73.9 MB/s eta 0:00:00


In [2]:
import json, os, sys, time, math, pickle, re, unicodedata, gc, hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Sequence
from collections import defaultdict, Counter
from datetime import datetime, timezone
import numpy as np
import torch

# === GPU Detection ===
if torch.cuda.is_available():
    DEVICE = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'🚀 GPU detected: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    DEVICE = 'cpu'
    print('⚠️ No GPU, using CPU (sẽ chậm hơn)')

print(f'Device: {DEVICE}')
print('Setup complete.')

🚀 GPU detected: Tesla T4 (14.6 GB)
Device: cuda
Setup complete.


## 2. Configuration

In [3]:
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    FAISS_DIR = Path('/kaggle/input/datasets/kittrntunk/faiss-chunk-meta')
    BM25_BASE_DIR = Path('/kaggle/input/datasets/nguyenlethienlyy/bm25-tokenized/bm25')
    QA_DIR = Path('/kaggle/input/datasets/phuongthao205/qa-legalrag/Benchmark')
    OUTPUT_BASE = Path('/kaggle/working/evaluation_runs/ablation3_reranker')
    CACHE_DIR = Path('/kaggle/working/retrieval_cache')
else:
    PROJECT_ROOT = Path('.').resolve()
    if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    FAISS_DIR = PROJECT_ROOT / 'data' / 'faiss_index'
    BM25_BASE_DIR = PROJECT_ROOT / 'data' / 'sparse_index'
    QA_DIR = PROJECT_ROOT / 'data' / 'benchmark'
    OUTPUT_BASE = PROJECT_ROOT / 'evaluation_runs' / 'ablation3_reranker'
    CACHE_DIR = PROJECT_ROOT / 'evaluation_runs' / 'retrieval_cache'

EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'
CROSS_ENCODER_MODEL = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
TOP_K = 30; TOP_N = 10
CE_CANDIDATE_MULT = 3; RRF_K = 60
SCORE_THRESHOLD = 0.30
TOP_K_EVAL = [1, 3, 5, 10]
ABLATION_LIMIT = None  # Set 5 để smoke test

LLM_BASE_URL = 'https://api.shopaikey.com/v1'
LLM_API_KEY = 'sk-5EAiA6CNDAmXwyewsIMXf4rsZWSkdAGijaEqBFKlWOCC954Z'
LLM_MODEL = 'gpt-4o-mini'
GEN_TEMPERATURE = 0.0; GEN_MAX_TOKENS = 1024; GEN_TIMEOUT = 60

print('Environment:', 'Kaggle' if IS_KAGGLE else 'Local')
print(f'Device: {DEVICE}')
print(f'Cache dir: {CACHE_DIR}')

Environment: Kaggle
Device: cuda
Cache dir: /kaggle/working/retrieval_cache


## 3. Verify Paths

In [4]:
def verify_path(path, desc):
    ok = path.exists()
    print(f'  {"✅" if ok else "❌"} {desc}: {path}')
    return ok

print('=== Kiểm tra paths ===')
ok1 = verify_path(FAISS_DIR, 'FAISS')
ok2 = verify_path(BM25_BASE_DIR, 'BM25')
ok3 = verify_path(QA_DIR, 'QA')

BENCHMARK_PATH = None
if ok3:
    for p in ['qa_final.jsonl', '*.jsonl']:
        found = list(QA_DIR.glob(p))
        if found:
            BENCHMARK_PATH = found[0]; break
    print(f'  📝 Benchmark: {BENCHMARK_PATH}')

DENSE_CACHE = CACHE_DIR / 'dense_all_hits.pkl'
BM25_CACHE = CACHE_DIR / 'bm25_all_hits.pkl'
print(f'  💾 Dense cache: {"✅ EXISTS" if DENSE_CACHE.exists() else "❌ not found"}')
print(f'  💾 BM25 cache:  {"✅ EXISTS" if BM25_CACHE.exists() else "❌ not found"}')

if not all([ok1, ok2, ok3, BENCHMARK_PATH]):
    print('\n⚠️ PATHS KHÔNG ĐÚNG!')
else:
    print('\n✅ OK!')

=== Kiểm tra paths ===
  ✅ FAISS: /kaggle/input/datasets/kittrntunk/faiss-chunk-meta
  ✅ BM25: /kaggle/input/datasets/nguyenlethienlyy/bm25-tokenized/bm25
  ✅ QA: /kaggle/input/datasets/phuongthao205/qa-legalrag/Benchmark
  📝 Benchmark: /kaggle/input/datasets/phuongthao205/qa-legalrag/Benchmark/qa_final.jsonl
  💾 Dense cache: ❌ not found
  💾 BM25 cache:  ❌ not found

✅ OK!


## 4. Schema, Metrics, Generator Utils

In [5]:
@dataclass(frozen=True)
class SearchHit:
    point_id: str
    score: float
    payload: dict[str, Any]

@dataclass(frozen=True)
class LatencyBreakdown:
    dense_latency_s: float = 0.0
    sparse_latency_s: float = 0.0
    fusion_latency_s: float = 0.0
    cross_encoder_latency_s: float = 0.0
    generation_latency_s: float = 0.0
    total_latency_s: float = 0.0
    def to_dict(self):
        return {k: round(v, 4) for k, v in {
            'dense_latency_s': self.dense_latency_s,
            'sparse_latency_s': self.sparse_latency_s,
            'fusion_latency_s': self.fusion_latency_s,
            'cross_encoder_latency_s': self.cross_encoder_latency_s,
            'generation_latency_s': self.generation_latency_s,
            'total_latency_s': self.total_latency_s,
        }.items()}

print('Schema defined.')

Schema defined.


In [6]:
def recall_at_k(ret, rel, k):
    return len(set(ret[:k]) & rel) / len(rel) if rel else 0.0
def hit_at_k(ret, rel, k):
    return 1.0 if rel and set(ret[:k]) & rel else 0.0
def mrr_at_k(ret, rel, k):
    if not rel: return 0.0
    for i, c in enumerate(ret[:k], 1):
        if c in rel: return 1.0/i
    return 0.0
def ndcg_at_k(ret, rel, k):
    if not rel: return 0.0
    dcg = sum(1.0/math.log2(i+1) for i, c in enumerate(ret[:k], 1) if c in rel)
    ideal = sum(1.0/math.log2(i+1) for i in range(1, min(len(rel), k)+1))
    return dcg/ideal if ideal else 0.0
def precision_at_k(ret, rel, k):
    return len(set(ret[:k]) & rel) / k if k else 0.0

_PUNCT_RE = re.compile(r'[^\w\s]', flags=re.UNICODE)
_SPACE_RE = re.compile(r'\s+')
def normalize_text(text):
    text = '' if text is None else str(text)
    text = unicodedata.normalize('NFC', text).lower()
    return _SPACE_RE.sub(' ', _PUNCT_RE.sub(' ', text)).strip()
def tokenize_text(text):
    n = normalize_text(text)
    return n.split() if n else []
def exact_match(pred, ref):
    return 1.0 if normalize_text(pred) == normalize_text(ref) else 0.0
def token_f1(pred, ref):
    pt, rt = tokenize_text(pred), tokenize_text(ref)
    if not pt and not rt: return 1.0
    if not pt or not rt: return 0.0
    common = sum((Counter(pt) & Counter(rt)).values())
    if common == 0: return 0.0
    p, r = common/len(pt), common/len(rt)
    return 2*p*r/(p+r)
def _lcs_len(a, b):
    prev = [0]*(len(b)+1)
    for ta in a:
        curr = [0]
        for j, tb in enumerate(b, 1):
            curr.append(prev[j-1]+1 if ta == tb else max(prev[j], curr[-1]))
        prev = curr
    return prev[-1]
def rouge_l(pred, ref):
    pt, rt = tokenize_text(pred), tokenize_text(ref)
    if not pt and not rt: return 1.0
    if not pt or not rt: return 0.0
    lcs = _lcs_len(pt, rt)
    p, r = lcs/len(pt), lcs/len(rt)
    return 2*p*r/(p+r) if (p+r) else 0.0
def is_unanswerable_text(text):
    n = normalize_text(text)
    return any(m in n for m in ['không có đủ thông tin','không đủ thông tin','không đủ căn cứ','không tìm thấy','không có thông tin'])
def aggregate_metrics(rows, keys):
    out = {'count': len(rows)}
    for k in keys:
        vals = [float(r[k]) for r in rows if r.get(k) is not None]
        out[k] = sum(vals)/len(vals) if vals else None
    return out
def aggregate_by(rows, field, keys):
    groups = defaultdict(list)
    for r in rows: groups[str(r.get(field) or 'unknown')].append(r)
    return {n: aggregate_metrics(g, keys) for n, g in sorted(groups.items())}

RET_METRIC_KEYS = [f'{n}@{k}' for k in TOP_K_EVAL for n in ['recall','hit','mrr','ndcg','precision']]
GEN_METRIC_KEYS = ['exact_match', 'token_f1', 'rouge_l', 'unanswerable_accuracy']
ALL_METRIC_KEYS = RET_METRIC_KEYS + GEN_METRIC_KEYS
print('Metrics defined.')

Metrics defined.


In [7]:
INSUFFICIENT_CONTEXT = 'Không có đủ thông tin trong ngữ cảnh được cung cấp.'
PROMPT_TEMPLATE = """You are a Vietnamese legal retrieval-augmented answering system.
Use only the supplied CONTEXT. Do not invent facts outside it.
If the context is insufficient, answer exactly:
"{insufficient}"

Answer the question directly from the context.

Output contract:
- Return only the final answer; never reveal hidden reasoning or chain-of-thought.
- For answer_type "boolean", the first line must be exactly "Có" or "Không";
  any explanation follows a line beginning "Giải thích:".
- For answer_type "unanswerable", return only the insufficient-context statement.
- Otherwise answer concisely in Vietnamese.
- Cite supporting context with [1], [2], etc. immediately after the claim.
- Do not add a bibliography. Cite only a source that supports the claim.

QUESTION:
{question}

ANSWER_TYPE:
{answer_type}

CONTEXT:
{context}

FINAL ANSWER:"""

_THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL | re.IGNORECASE)
_CITE_RE = re.compile(r'\[\s*(\d+)\s*\]')

def format_chunks_as_context(hits):
    blocks = []
    for i, hit in enumerate(hits, 1):
        p = hit.payload
        lines = [f'[SOURCE {i}]']
        for label, key in [('Title','title'),('Article','article_number'),('Section','citation_anchor'),('Chunk ID','chunk_id')]:
            v = p.get(key)
            if v: lines.append(f'{label}: {v}')
        lines.extend(['Content:', str(p.get('chunk_text') or p.get('text') or '')[:4000]])
        blocks.append('\n'.join(lines))
    return '\n\n'.join(blocks)

def build_prompt(question, answer_type, hits):
    return PROMPT_TEMPLATE.format(insufficient=INSUFFICIENT_CONTEXT, question=question.strip(),
                                  answer_type=(answer_type or '').strip(), context=format_chunks_as_context(hits))
def parse_answer(raw_text):
    return _THINK_RE.sub('', raw_text or '').strip()
def count_citations(text):
    return len(set(_CITE_RE.findall(text or '')))

print('Generator utils defined.')

Generator utils defined.


In [8]:
from openai import OpenAI
gen_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

def call_llm(prompt, *, temperature=GEN_TEMPERATURE, max_tokens=GEN_MAX_TOKENS):
    resp = gen_client.chat.completions.create(
        model=LLM_MODEL, messages=[{'role':'user','content':prompt}],
        temperature=temperature, max_tokens=max_tokens, timeout=GEN_TIMEOUT)
    return (resp.choices[0].message.content or '').strip()

print('Testing LLM...')
print(f'  Response: {call_llm("1+1 bằng mấy?")[:100]}')
print('✅ Generator ready.')

Testing LLM...
  Response: 1 + 1 bằng 2.
✅ Generator ready.


## 5. Load QA Benchmark

In [9]:
print(f'Loading benchmark from {BENCHMARK_PATH} ...')
qa_data = []
with open(BENCHMARK_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line: qa_data.append(json.loads(line))

eval_qa, skip_no_gt = [], 0
for qa in qa_data:
    gt = qa.get('ground_truth') or {}
    gt_chunks = {str(c) for c in gt.get('chunk_ids') or [] if c}
    at = str(qa.get('answer_type') or '').lower()
    cat = str(qa.get('category') or '').lower()
    is_unanswerable = (at == 'unanswerable' or cat == 'unanswerable')
    if not gt_chunks and not is_unanswerable:
        skip_no_gt += 1; continue
    eval_qa.append((qa, gt_chunks, is_unanswerable))

if ABLATION_LIMIT: eval_qa = eval_qa[:ABLATION_LIMIT]
all_questions = [str(qa.get('question') or '') for qa, _, _ in eval_qa]
n_unans = sum(1 for _,_,u in eval_qa if u)
print(f'Eval: {len(eval_qa)} total ({len(eval_qa)-n_unans} answerable + {n_unans} unanswerable)')

Loading benchmark from /kaggle/input/datasets/phuongthao205/qa-legalrag/Benchmark/qa_final.jsonl ...
Eval: 500 total (400 answerable + 100 unanswerable)


## 6. Load/Compute Dense + BM25 (with cache)

**Lần đầu**: chạy Dense (GPU) + BM25 → **lưu pickle cache**  
**Lần sau**: load cache → **skip hoàn toàn**

In [10]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _hits_to_dicts(hits_list):
    return [[(h.point_id, h.score, h.payload) for h in hits] for hits in hits_list]
def _dicts_to_hits(data):
    return [[SearchHit(pid, sc, pay) for pid, sc, pay in hits] for hits in data]

# Try loading cache
if DENSE_CACHE.exists() and BM25_CACHE.exists():
    print('💾 Loading from cache...')
    t0 = time.perf_counter()
    with open(DENSE_CACHE, 'rb') as f:
        cache_dense = pickle.load(f)
    dense_all_hits = _dicts_to_hits(cache_dense['hits'])
    dense_all_latencies = cache_dense['latencies']
    print(f'  ✅ Dense: {len(dense_all_hits)} queries ({time.perf_counter()-t0:.1f}s)')

    t0 = time.perf_counter()
    with open(BM25_CACHE, 'rb') as f:
        cache_bm25 = pickle.load(f)
    bm25_all_hits = _dicts_to_hits(cache_bm25['hits'])
    print(f'  ✅ BM25:  {len(bm25_all_hits)} queries ({time.perf_counter()-t0:.1f}s)')

    if len(dense_all_hits) != len(eval_qa) or len(bm25_all_hits) != len(eval_qa):
        print(f'  ⚠️ Cache size mismatch! Cache={len(dense_all_hits)}, Eval={len(eval_qa)}')
        print(f'  → Will re-compute.')
        dense_all_hits = None
    else:
        print(f'\n✅ Cache valid! Skip retrieval → straight to reranking.')
else:
    dense_all_hits = None
    print('❌ No cache → will compute Dense + BM25 from scratch.')

❌ No cache → will compute Dense + BM25 from scratch.


In [11]:
# === DENSE: compute if no cache ===
if dense_all_hits is None:
    import faiss
    from sentence_transformers import SentenceTransformer

    print('Loading FAISS index...')
    t0 = time.perf_counter()
    faiss_index = faiss.read_index(str(FAISS_DIR / 'index.faiss'))
    
    # Move FAISS to GPU if available
    if DEVICE == 'cuda':
        try:
            res = faiss.StandardGpuResources()
            faiss_index = faiss.index_cpu_to_gpu(res, 0, faiss_index)
            print(f'  🚀 FAISS on GPU ({faiss_index.ntotal:,} vectors, {time.perf_counter()-t0:.1f}s)')
        except Exception as e:
            print(f'  ⚠️ FAISS GPU failed ({e}), using CPU')
            print(f'  {faiss_index.ntotal:,} vectors ({time.perf_counter()-t0:.1f}s)')
    else:
        print(f'  {faiss_index.ntotal:,} vectors ({time.perf_counter()-t0:.1f}s)')

    print('Loading payloads...')
    t0 = time.perf_counter()
    dense_payloads = {}
    with open(FAISS_DIR / 'payloads.jsonl', 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if line: dense_payloads[i] = json.loads(line)
    print(f'  {len(dense_payloads):,} payloads ({time.perf_counter()-t0:.1f}s)')

    id_map_path = FAISS_DIR / 'id_map.json'
    if id_map_path.exists():
        with open(id_map_path, 'r', encoding='utf-8') as f: raw = json.load(f)
        dense_id_map = {int(v): str(k) for k, v in raw.items()}
    else:
        dense_id_map = {i: str(dense_payloads.get(i, {}).get('chunk_id', i)) for i in dense_payloads}

    print(f'Loading {EMBEDDING_MODEL} on {DEVICE}...')
    t0 = time.perf_counter()
    embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
    print(f'  🚀 Loaded on {DEVICE} in {time.perf_counter()-t0:.1f}s')

    def dense_search(query, *, top_k=30, score_threshold=0.0):
        vec = embedder.encode(['query: ' + query], normalize_embeddings=True)
        qv = np.array(vec, dtype=np.float32)
        limit = min(top_k * 3, faiss_index.ntotal)
        scores, indices = faiss_index.search(qv, limit)
        hits = []
        for sc, idx in zip(scores[0], indices[0]):
            if idx < 0 or float(sc) < score_threshold: continue
            hits.append(SearchHit(point_id=dense_id_map.get(int(idx), str(idx)),
                                  score=float(sc), payload=dense_payloads.get(int(idx), {})))
            if len(hits) >= top_k: break
        return hits

    print(f'\nPre-computing dense for {len(eval_qa)} queries on {DEVICE}...')
    dense_all_hits, dense_all_latencies = [], []
    dense_t0 = time.perf_counter()
    for qi in range(len(eval_qa)):
        t0 = time.perf_counter()
        hits = dense_search(all_questions[qi], top_k=TOP_K, score_threshold=SCORE_THRESHOLD)
        dense_all_hits.append(hits)
        dense_all_latencies.append(time.perf_counter() - t0)
        if (qi+1) % 100 == 0:
            print(f'  {qi+1}/{len(eval_qa)} ({time.perf_counter()-dense_t0:.0f}s)')
    print(f'✅ Dense done in {time.perf_counter()-dense_t0:.0f}s')

    with open(DENSE_CACHE, 'wb') as f:
        pickle.dump({'hits': _hits_to_dicts(dense_all_hits), 'latencies': dense_all_latencies}, f)
    print(f'💾 Dense cache saved ({DENSE_CACHE.stat().st_size/1024/1024:.1f} MB)')

    del faiss_index, dense_payloads, dense_id_map, embedder
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    gc.collect()
else:
    print('⏭️ Dense: using cache')

Loading FAISS index...
  🚀 FAISS on GPU (1,513,376 vectors, 35.6s)
Loading payloads...
  1,513,376 payloads (99.4s)
Loading intfloat/multilingual-e5-large on cuda...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

  🚀 Loaded on cuda in 21.0s

Pre-computing dense for 500 queries on cuda...
  100/500 (6s)
  200/500 (11s)
  300/500 (16s)
  400/500 (21s)
  500/500 (26s)
✅ Dense done in 26s
💾 Dense cache saved (40.9 MB)


In [12]:
# === BM25: compute if no cache ===
if not BM25_CACHE.exists():
    def simple_tokenize(text):
        text = unicodedata.normalize('NFC', text).lower()
        text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
        return [t for t in text.split() if len(t) > 1]
    try:
        from underthesea import word_tokenize as _ws
        def bm25_tokenize(text): return simple_tokenize(_ws(text, format='text'))
        print('✅ underthesea tokenizer')
    except ImportError:
        bm25_tokenize = simple_tokenize
        print('⚠️ simple tokenizer')

    all_tokenized = [bm25_tokenize(q) for q in all_questions]
    shard_dirs = sorted([d for d in BM25_BASE_DIR.iterdir()
                         if d.is_dir() and d.name.startswith('shard_')])
    print(f'Found {len(shard_dirs)} shards, {len(all_tokenized)} queries')

    bm25_all_hits = [[] for _ in range(len(eval_qa))]
    bm25_t0 = time.perf_counter()

    for shard_dir in shard_dirs:
        st0 = time.perf_counter()
        idx_p, meta_p = shard_dir / 'bm25_index.pkl', shard_dir / 'bm25_metadata.pkl'
        if not idx_p.exists() or not meta_p.exists():
            print(f'  ⚠️ Skip {shard_dir.name}'); continue
        with idx_p.open('rb') as f: bm25 = pickle.load(f)
        with meta_p.open('rb') as f: meta = pickle.load(f)
        chunk_ids = meta['chunk_ids']
        shard_pays = meta.get('payloads', [{}]*len(chunk_ids))
        del meta
        load_t = time.perf_counter() - st0

        s_t0 = time.perf_counter()
        for qi, tok_q in enumerate(all_tokenized):
            scores = bm25.get_scores(tok_q)
            top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:TOP_K]
            for idx in top_idx:
                sc = float(scores[idx])
                if sc <= 0: break
                cid = chunk_ids[idx]
                pay = shard_pays[idx] if idx < len(shard_pays) and isinstance(shard_pays[idx], dict) else {'chunk_id': cid}
                bm25_all_hits[qi].append(SearchHit(point_id=cid, score=sc, payload=pay))
        print(f'  ✅ {shard_dir.name}: {len(chunk_ids):,} docs | load={load_t:.1f}s | search={time.perf_counter()-s_t0:.1f}s')
        del bm25, chunk_ids, shard_pays; gc.collect()

    for qi in range(len(eval_qa)):
        bm25_all_hits[qi].sort(key=lambda h: h.score, reverse=True)
        seen, deduped = set(), []
        for h in bm25_all_hits[qi]:
            cid = str(h.payload.get('chunk_id') or h.point_id)
            if cid not in seen: seen.add(cid); deduped.append(h)
            if len(deduped) >= TOP_K: break
        bm25_all_hits[qi] = deduped

    print(f'\n✅ BM25 done in {time.perf_counter()-bm25_t0:.0f}s')
    with open(BM25_CACHE, 'wb') as f:
        pickle.dump({'hits': _hits_to_dicts(bm25_all_hits)}, f)
    print(f'💾 BM25 cache saved ({BM25_CACHE.stat().st_size/1024/1024:.1f} MB)')
else:
    print('⏭️ BM25: using cache')

print(f'\n📦 Ready: dense[{len(dense_all_hits)}] + bm25[{len(bm25_all_hits)}]')

✅ underthesea tokenizer
Found 10 shards, 500 queries
  ✅ shard_00: 151,338 docs | load=6.5s | search=1273.1s
  ✅ shard_01: 151,338 docs | load=6.8s | search=1294.8s
  ✅ shard_02: 151,336 docs | load=7.7s | search=1325.6s
  ✅ shard_03: 151,338 docs | load=7.3s | search=1288.8s
  ✅ shard_04: 151,338 docs | load=6.1s | search=1249.3s
  ✅ shard_05: 151,338 docs | load=6.6s | search=1264.9s
  ✅ shard_06: 151,338 docs | load=7.8s | search=1265.7s
  ✅ shard_07: 151,338 docs | load=8.3s | search=1284.8s
  ✅ shard_08: 151,338 docs | load=8.1s | search=1302.8s
  ✅ shard_09: 151,334 docs | load=7.6s | search=1310.0s

✅ BM25 done in 12943s
💾 BM25 cache saved (32.1 MB)

📦 Ready: dense[500] + bm25[500]


## 7. Load Cross-Encoder (GPU)

In [13]:
from sentence_transformers import CrossEncoder
print(f'Loading Cross-Encoder on {DEVICE}: {CROSS_ENCODER_MODEL} ...')
t0 = time.perf_counter()
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, device=DEVICE)
print(f'  🚀 Loaded on {DEVICE} in {time.perf_counter()-t0:.1f}s')
print('✅ Cross-Encoder ready.')

Loading Cross-Encoder on cuda: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1 ...


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  🚀 Loaded on cuda in 7.9s
✅ Cross-Encoder ready.


## 8. Fusion & Reranking Functions

In [14]:
def hybrid_merge_score(dense_hits, bm25_hits, top_n=10):
    d_max = max((h.score for h in dense_hits), default=1.0) or 1.0
    b_max = max((h.score for h in bm25_hits), default=1.0) or 1.0
    combined = {}
    for h in dense_hits:
        cid = str(h.payload.get('chunk_id') or h.point_id)
        n = h.score / d_max
        if cid not in combined or n > combined[cid].score:
            combined[cid] = SearchHit(h.point_id, n, h.payload)
    for h in bm25_hits:
        cid = str(h.payload.get('chunk_id') or h.point_id)
        n = h.score / b_max
        if cid not in combined or n > combined[cid].score:
            combined[cid] = SearchHit(h.point_id, n, h.payload)
    return sorted(combined.values(), key=lambda h: h.score, reverse=True)[:top_n]

def rrf_fusion(dense_hits, bm25_hits, *, k=60, top_n=10):
    rrf_scores, best_hit = {}, {}
    for rank, h in enumerate(dense_hits, 1):
        cid = str(h.payload.get('chunk_id') or h.point_id)
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0/(k+rank)
        if cid not in best_hit: best_hit[cid] = h
    for rank, h in enumerate(bm25_hits, 1):
        cid = str(h.payload.get('chunk_id') or h.point_id)
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0/(k+rank)
        if cid not in best_hit: best_hit[cid] = h
    sorted_ids = sorted(rrf_scores, key=lambda c: rrf_scores[c], reverse=True)
    return [SearchHit(best_hit[c].point_id, rrf_scores[c], best_hit[c].payload) for c in sorted_ids[:top_n]]

def cross_encoder_rerank(query, hits, *, top_n=10):
    if not hits: return [], 0.0
    t0 = time.perf_counter()
    pairs = [(query, str(h.payload.get('chunk_text') or '')) for h in hits]
    ce_scores = cross_encoder.predict(pairs)
    scored = [SearchHit(h.point_id, float(s), h.payload) for h, s in zip(hits, ce_scores)]
    scored.sort(key=lambda h: h.score, reverse=True)
    return scored[:top_n], time.perf_counter() - t0

print('Fusion & reranking defined.')

Fusion & reranking defined.


## 9. Ablation Configs

In [15]:
CONFIGS = {
    'Rerank-None-Hybrid':             {'description': 'Score merge, no reranking',         'use_rrf': False, 'use_cross_encoder': False},
    'Rerank-RRF-Hybrid':              {'description': 'RRF fusion',                        'use_rrf': True,  'use_cross_encoder': False},
    'Rerank-CrossEncoder-Hybrid':     {'description': 'Score merge + CrossEncoder rerank',  'use_rrf': False, 'use_cross_encoder': True},
    'Rerank-RRF+CrossEncoder-Hybrid': {'description': 'RRF + CrossEncoder rerank',          'use_rrf': True,  'use_cross_encoder': True},
}
print(f'{len(CONFIGS)} configs × {len(eval_qa)} queries')
print(f'⚡ Dense+BM25 cached → chỉ fusion/rerank/generation')
print(f'🚀 Cross-Encoder on {DEVICE}')

4 configs × 500 queries
⚡ Dense+BM25 cached → chỉ fusion/rerank/generation
🚀 Cross-Encoder on cuda


## 10. Run Ablation

In [16]:
all_results = {}
grand_t0 = time.perf_counter()

for cfg_name, cfg in CONFIGS.items():
    print(f'\n{"="*70}\n{cfg_name} (rrf={cfg["use_rrf"]}, ce={cfg["use_cross_encoder"]})\n{"="*70}')
    use_rrf, use_ce = cfg['use_rrf'], cfg['use_cross_encoder']
    cases, latencies, gen_errors = [], [], 0
    run_t0 = time.perf_counter()

    for qi, (qa, gt_chunks, is_unanswerable) in enumerate(eval_qa):
        question = all_questions[qi]
        qa_id = str(qa.get('qa_id') or qa.get('id') or f'qa_{qi+1}')
        answer_type = str(qa.get('answer_type') or '')
        ref_answer = str(qa.get('reference_answer') or qa.get('answer') or '')
        try:
            t0 = time.perf_counter()
            d_hits, b_hits = dense_all_hits[qi], bm25_all_hits[qi]
            d_lat = dense_all_latencies[qi] if 'dense_all_latencies' in dir() else 0.0

            f_t0 = time.perf_counter()
            n_cand = TOP_N * CE_CANDIDATE_MULT if use_ce else TOP_N
            fused = rrf_fusion(d_hits, b_hits, k=RRF_K, top_n=n_cand) if use_rrf else hybrid_merge_score(d_hits, b_hits, top_n=n_cand)
            f_lat = time.perf_counter() - f_t0

            ce_lat = 0.0
            if use_ce:
                fused, ce_lat = cross_encoder_rerank(question, fused, top_n=TOP_N)
            else:
                fused = fused[:TOP_N]

            hits = fused
            ret_lat = LatencyBreakdown(dense_latency_s=d_lat, fusion_latency_s=f_lat,
                                       cross_encoder_latency_s=ce_lat, total_latency_s=time.perf_counter()-t0)
            ret_ids = [str(h.payload.get('chunk_id') or h.point_id) for h in hits]

            gen_lat, predicted, citation_count, gen_error = 0.0, '', 0, None
            if hits:
                try:
                    gen_t0 = time.perf_counter()
                    predicted = parse_answer(call_llm(build_prompt(question, answer_type, hits)))
                    gen_lat = time.perf_counter() - gen_t0
                    citation_count = count_citations(predicted)
                except Exception as e:
                    gen_error = str(e); gen_errors += 1
                    if gen_errors <= 3: print(f'  ⚠️ Gen error {qa_id}: {str(e)[:100]}')
            else:
                predicted = INSUFFICIENT_CONTEXT

            row = {'qa_id': qa_id, 'question': question, 'category': qa.get('category'),
                   'difficulty': qa.get('difficulty'), 'answer_type': answer_type,
                   'is_unanswerable': is_unanswerable, 'reference_answer': ref_answer,
                   'predicted_answer': predicted, 'ground_truth_chunk_ids': sorted(gt_chunks),
                   'retrieved_chunk_ids': ret_ids, 'num_retrieved': len(ret_ids),
                   'citation_count': citation_count, 'generation_error': gen_error}

            if gt_chunks:
                for k in TOP_K_EVAL:
                    row[f'recall@{k}'] = recall_at_k(ret_ids, gt_chunks, k)
                    row[f'hit@{k}'] = hit_at_k(ret_ids, gt_chunks, k)
                    row[f'mrr@{k}'] = mrr_at_k(ret_ids, gt_chunks, k)
                    row[f'ndcg@{k}'] = ndcg_at_k(ret_ids, gt_chunks, k)
                    row[f'precision@{k}'] = precision_at_k(ret_ids, gt_chunks, k)

            if is_unanswerable:
                row['exact_match'] = row['token_f1'] = row['rouge_l'] = None
                row['unanswerable_accuracy'] = 1.0 if is_unanswerable_text(predicted) else 0.0
            else:
                if predicted and not gen_error:
                    pe = predicted.split('\n')[0].strip() if answer_type=='boolean' else predicted
                    re_ = ref_answer.split('\n')[0].strip() if answer_type=='boolean' else ref_answer
                    row['exact_match'] = exact_match(pe, re_)
                    row['token_f1'] = token_f1(predicted, ref_answer)
                    row['rouge_l'] = rouge_l(predicted, ref_answer)
                else:
                    row['exact_match'] = row['token_f1'] = row['rouge_l'] = 0.0
                row['unanswerable_accuracy'] = 1.0 if not is_unanswerable_text(predicted) else 0.0

            cases.append(row)
            latencies.append(LatencyBreakdown(dense_latency_s=ret_lat.dense_latency_s,
                fusion_latency_s=ret_lat.fusion_latency_s, cross_encoder_latency_s=ret_lat.cross_encoder_latency_s,
                generation_latency_s=gen_lat, total_latency_s=ret_lat.total_latency_s+gen_lat).to_dict())
        except Exception as e:
            print(f'  ❌ FATAL {qa_id}: {e}')
            cases.append({'qa_id': qa_id, 'error': str(e)})
            latencies.append(LatencyBreakdown().to_dict())

        if (qi+1) % 25 == 0:
            el = time.perf_counter()-run_t0; eta = (el/(qi+1))*(len(eval_qa)-qi-1)
            print(f'  {qi+1}/{len(eval_qa)} ({el:.0f}s, ~{eta:.0f}s left)')

    dur = time.perf_counter()-run_t0
    valid = [c for c in cases if 'error' not in c]
    summary = {
        'config_name': cfg_name, 'config': cfg,
        'counts': {'total': len(cases), 'evaluated': len(valid), 'errors': len(cases)-len(valid), 'gen_errors': gen_errors},
        'overall': aggregate_metrics(valid, ALL_METRIC_KEYS),
        'by_category': aggregate_by(valid, 'category', ALL_METRIC_KEYS),
        'by_difficulty': aggregate_by(valid, 'difficulty', ALL_METRIC_KEYS),
        'by_answer_type': aggregate_by(valid, 'answer_type', ALL_METRIC_KEYS),
        'latency': {
            'total_run_time_s': round(dur, 2),
            'avg': {k: round(np.mean([l[k] for l in latencies]), 4) for k in latencies[0]} if latencies else {},
            'median': {k: round(float(np.median([l[k] for l in latencies])), 4) for k in latencies[0]} if latencies else {},
        },
    }
    print(f'\n  ✅ {cfg_name}: {len(valid)} cases, {dur:.0f}s')
    for m in ['recall@10','mrr@10','ndcg@10','token_f1','rouge_l','exact_match']:
        v = summary['overall'].get(m)
        print(f'     {m}: {v:.4f}' if v is not None else f'     {m}: N/A')
    all_results[cfg_name] = (cases, latencies, summary)

grand_total = time.perf_counter()-grand_t0
print(f'\n{"="*70}\n✅ All done! {grand_total:.0f}s ({grand_total/60:.1f}min)\n{"="*70}')


Rerank-None-Hybrid (rrf=False, ce=False)
  25/500 (88s, ~1663s left)
  50/500 (181s, ~1632s left)
  75/500 (334s, ~1893s left)
  100/500 (436s, ~1743s left)
  125/500 (545s, ~1636s left)
  150/500 (618s, ~1441s left)
  175/500 (706s, ~1310s left)
  200/500 (781s, ~1172s left)
  225/500 (855s, ~1045s left)
  250/500 (954s, ~954s left)
  275/500 (1040s, ~851s left)
  300/500 (1145s, ~763s left)
  325/500 (1236s, ~666s left)
  350/500 (1324s, ~567s left)
  375/500 (1412s, ~471s left)
  400/500 (1517s, ~379s left)
  425/500 (1634s, ~288s left)
  450/500 (1734s, ~193s left)
  475/500 (1837s, ~97s left)
  500/500 (1926s, ~0s left)

  ✅ Rerank-None-Hybrid: 500 cases, 1926s
     recall@10: 0.4842
     mrr@10: 0.2167
     ndcg@10: 0.2714
     token_f1: 0.2735
     rouge_l: 0.2379
     exact_match: 0.1450

Rerank-RRF-Hybrid (rrf=True, ce=False)
  25/500 (87s, ~1648s left)
  50/500 (165s, ~1486s left)
  75/500 (260s, ~1472s left)
  100/500 (377s, ~1507s left)
  125/500 (508s, ~1524s left)
  150/

## 11. Save Results

In [17]:
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
for cfg_name, (cases, latencies, summary) in all_results.items():
    cfg_dir = OUTPUT_BASE / cfg_name; cfg_dir.mkdir(parents=True, exist_ok=True)
    with open(cfg_dir/'retrieval_cases.jsonl','w',encoding='utf-8') as f:
        for c in cases: json.dump(c,f,ensure_ascii=False,default=str); f.write('\n')
    with open(cfg_dir/'retrieval_metrics.json','w',encoding='utf-8') as f:
        json.dump(summary,f,indent=2,ensure_ascii=False,default=str)
    with open(cfg_dir/'latency.json','w',encoding='utf-8') as f:
        json.dump(summary['latency'],f,indent=2,ensure_ascii=False)
    with open(cfg_dir/'manifest.json','w',encoding='utf-8') as f:
        json.dump({'config':cfg_name,'timestamp':datetime.now(timezone.utc).isoformat(),
                   'eval_count':len(cases),'embedding_model':EMBEDDING_MODEL,
                   'cross_encoder_model':CROSS_ENCODER_MODEL,'generator_model':LLM_MODEL,
                   'top_k':TOP_K,'top_n':TOP_N,'rrf_k':RRF_K,'ce_candidate_mult':CE_CANDIDATE_MULT,
                   'device': DEVICE},
                  f,indent=2,ensure_ascii=False)
    print(f'  ✅ {cfg_name}')

combined = {'owner':'Người 3 (Kiệt)','ablation':'Ablation 3: Reranker',
    'timestamp':datetime.now(timezone.utc).isoformat(),'device': DEVICE,
    'settings':{'embedding_model':EMBEDDING_MODEL,'cross_encoder_model':CROSS_ENCODER_MODEL,
                'generator_model':LLM_MODEL,'top_k':TOP_K,'top_n':TOP_N,'rrf_k':RRF_K,
                'ce_candidate_mult':CE_CANDIDATE_MULT,'eval_count':len(eval_qa)},
    'configs':{n:{'overall':s['overall'],'latency':s['latency'],'by_category':s['by_category'],
                  'by_answer_type':s['by_answer_type']} for n,(_,_,s) in all_results.items()}}
with open(OUTPUT_BASE/'summary.json','w',encoding='utf-8') as f:
    json.dump(combined,f,indent=2,ensure_ascii=False,default=str)
print(f'\n✅ summary.json saved')

  ✅ Rerank-None-Hybrid
  ✅ Rerank-RRF-Hybrid
  ✅ Rerank-CrossEncoder-Hybrid
  ✅ Rerank-RRF+CrossEncoder-Hybrid

✅ summary.json saved


## 12. Comparison Table

In [18]:
import csv
csv_path = OUTPUT_BASE / 'comparison.csv'
fieldnames = ['Config'] + RET_METRIC_KEYS + GEN_METRIC_KEYS + ['avg_ce_latency','avg_gen_latency','avg_total_latency']
rows_csv = []
for name,(_,_,s) in all_results.items():
    row = {'Config': name}
    for k in RET_METRIC_KEYS+GEN_METRIC_KEYS:
        v = s['overall'].get(k); row[k] = round(v,4) if v is not None else ''
    row['avg_ce_latency'] = s['latency']['avg'].get('cross_encoder_latency_s',0)
    row['avg_gen_latency'] = s['latency']['avg'].get('generation_latency_s',0)
    row['avg_total_latency'] = s['latency']['avg'].get('total_latency_s',0)
    rows_csv.append(row)
with open(csv_path,'w',newline='',encoding='utf-8') as f:
    w = csv.DictWriter(f,fieldnames=fieldnames); w.writeheader(); w.writerows(rows_csv)

print('='*100)
print('ABLATION 3: RERANKER COMPARISON')
print('='*100)
print(f'{"Config":<40} {"R@10":>6} {"MRR@10":>7} {"nDCG@10":>8} {"F1":>6} {"RL":>6} {"EM":>6} {"CE_lat":>7} {"Tot_lat":>8}')
print('-'*100)
for name,(_,_,s) in all_results.items():
    o,lat = s['overall'], s['latency']['avg']
    print(f'{name:<40} {(o.get("recall@10",0) or 0):>6.4f} {(o.get("mrr@10",0) or 0):>7.4f} '
          f'{(o.get("ndcg@10",0) or 0):>8.4f} {(o.get("token_f1",0) or 0):>6.4f} '
          f'{(o.get("rouge_l",0) or 0):>6.4f} {(o.get("exact_match",0) or 0):>6.4f} '
          f'{lat.get("cross_encoder_latency_s",0):>7.4f} {lat.get("total_latency_s",0):>8.4f}')
print('='*100)

ABLATION 3: RERANKER COMPARISON
Config                                     R@10  MRR@10  nDCG@10     F1     RL     EM  CE_lat  Tot_lat
----------------------------------------------------------------------------------------------------
Rerank-None-Hybrid                       0.4842  0.2167   0.2714 0.2735 0.2379 0.1450  0.0000   3.8507
Rerank-RRF-Hybrid                        0.4382  0.2335   0.2728 0.2735 0.2400 0.1550  0.0000   3.9679
Rerank-CrossEncoder-Hybrid               0.5005  0.2818   0.3221 0.2700 0.2356 0.1525  0.3899   4.0756
Rerank-RRF+CrossEncoder-Hybrid           0.4640  0.2731   0.3099 0.2719 0.2363 0.1350  0.4018   4.6171


## 13. Visualization

In [19]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

cn = list(all_results.keys()); sn = ['None','RRF','CE','RRF+CE']
colors = ['#4A90D9','#E8833A','#50B86C','#D94A6B']
gm = lambda n,m: all_results[n][2]['overall'].get(m,0) or 0

fig, axes = plt.subplots(2,3,figsize=(18,10))
fig.suptitle('Ablation 3: Reranker Comparison (GPU-accelerated)', fontsize=16, fontweight='bold')

for ax,metric in zip([axes[0,0],axes[0,1],axes[0,2]], ['recall','mrr','ndcg']):
    for i,n in enumerate(cn):
        ax.plot(TOP_K_EVAL,[gm(n,f'{metric}@{k}') for k in TOP_K_EVAL],'o-',label=sn[i],color=colors[i],lw=2,ms=7)
    ax.set_title(f'{metric.upper()}@k'); ax.legend(fontsize=8); ax.grid(True,alpha=0.3); ax.set_xticks(TOP_K_EVAL)

ax=axes[1,0]; x=np.arange(3); w=0.18
for i,n in enumerate(cn):
    ax.bar(x+i*w,[gm(n,m) for m in ['exact_match','token_f1','rouge_l']],w,label=sn[i],color=colors[i])
ax.set_title('Generation Metrics'); ax.set_xticks(x+w*1.5); ax.set_xticklabels(['EM','F1','RL']); ax.legend(fontsize=8)

ax=axes[1,1]; bottom=np.zeros(4)
for lk,ll,lc in zip(['dense_latency_s','fusion_latency_s','cross_encoder_latency_s','generation_latency_s'],
                     ['Dense','Fusion','CE','Gen'],['#4A90D9','#E8833A','#D94A6B','#50B86C']):
    vs=[all_results[n][2]['latency']['avg'].get(lk,0) for n in cn]
    ax.bar(sn,vs,bottom=bottom,label=ll,color=lc); bottom+=np.array(vs)
ax.set_title('Latency Breakdown'); ax.legend(fontsize=8)

ax=axes[1,2]; x=np.arange(5); w=0.18
for i,n in enumerate(cn):
    ax.bar(x+i*w,[gm(n,m) for m in ['recall@10','mrr@10','ndcg@10','token_f1','rouge_l']],w,label=sn[i],color=colors[i])
ax.set_title('Key Metrics'); ax.set_xticks(x+w*1.5); ax.set_xticklabels(['R@10','MRR','nDCG','F1','RL']); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_BASE/'charts.png',dpi=150,bbox_inches='tight'); plt.show()
print('✅ Charts saved')

✅ Charts saved


## 14. Detailed Analysis

In [20]:
for breakdown_name, breakdown_key in [('PER-CATEGORY','by_category'),('PER-ANSWER-TYPE','by_answer_type')]:
    print(f'\n{"="*110}\n{breakdown_name}\n{"="*110}')
    groups = set()
    for _,(_,_,s) in all_results.items(): groups.update(s[breakdown_key].keys())
    for g in sorted(groups):
        print(f'\n--- {g.upper()} ---')
        print(f'{"Config":<40} {"R@10":>6} {"MRR@10":>7} {"nDCG@10":>8} {"F1":>6} {"RL":>6} {"Count":>6}')
        for name,(_,_,s) in all_results.items():
            d = s[breakdown_key].get(g, {})
            if not d: continue
            print(f'{name:<40} {(d.get("recall@10",0) or 0):>6.4f} {(d.get("mrr@10",0) or 0):>7.4f} '
                  f'{(d.get("ndcg@10",0) or 0):>8.4f} {(d.get("token_f1",0) or 0):>6.4f} '
                  f'{(d.get("rouge_l",0) or 0):>6.4f} {d.get("count",0):>6}')


PER-CATEGORY

--- CITATION ---
Config                                     R@10  MRR@10  nDCG@10     F1     RL  Count
Rerank-None-Hybrid                       0.6264  0.2175   0.3140 0.3151 0.2743    106
Rerank-RRF-Hybrid                        0.5557  0.2680   0.3300 0.3200 0.2815    106
Rerank-CrossEncoder-Hybrid               0.6594  0.3366   0.4058 0.3134 0.2776    106
Rerank-RRF+CrossEncoder-Hybrid           0.5557  0.3073   0.3629 0.3039 0.2635    106

--- CROSS_DOCUMENT ---
Config                                     R@10  MRR@10  nDCG@10     F1     RL  Count
Rerank-None-Hybrid                       0.2530  0.3136   0.2352 0.3030 0.2612     12
Rerank-RRF-Hybrid                        0.2758  0.3371   0.2248 0.3254 0.2686     12
Rerank-CrossEncoder-Hybrid               0.1621  0.1746   0.1266 0.2972 0.2505     12
Rerank-RRF+CrossEncoder-Hybrid           0.1848  0.1631   0.1340 0.2962 0.2434     12

--- LEGAL_VALIDITY ---
Config                                     R@10  MRR@10  nDC

In [21]:
print(f'\n{"="*70}\nABLATION 3: RERANKER — FINAL SUMMARY\n{"="*70}')
print(f'Runtime: {grand_total:.0f}s ({grand_total/60:.1f}min) | Queries: {len(eval_qa)} | Configs: {len(CONFIGS)}')
print(f'Device: {DEVICE}')
print(f'Models: {EMBEDDING_MODEL} | {CROSS_ENCODER_MODEL} | {LLM_MODEL}')
print(f'\n--- Best by metric ---')
for mn,mk in [('Recall@10','recall@10'),('MRR@10','mrr@10'),('nDCG@10','ndcg@10'),
              ('Token F1','token_f1'),('ROUGE-L','rouge_l'),('Exact Match','exact_match')]:
    bn = max(all_results, key=lambda n: all_results[n][2]['overall'].get(mk,0) or 0)
    bv = all_results[bn][2]['overall'].get(mk,0) or 0
    print(f'  {mn:<15}: {bn} ({bv:.4f})')
print(f'\n✅ Output: {OUTPUT_BASE}')


ABLATION 3: RERANKER — FINAL SUMMARY
Runtime: 8257s (137.6min) | Queries: 500 | Configs: 4
Device: cuda
Models: intfloat/multilingual-e5-large | cross-encoder/mmarco-mMiniLMv2-L12-H384-v1 | gpt-4o-mini

--- Best by metric ---
  Recall@10      : Rerank-CrossEncoder-Hybrid (0.5005)
  MRR@10         : Rerank-CrossEncoder-Hybrid (0.2818)
  nDCG@10        : Rerank-CrossEncoder-Hybrid (0.3221)
  Token F1       : Rerank-RRF-Hybrid (0.2735)
  ROUGE-L        : Rerank-RRF-Hybrid (0.2400)
  Exact Match    : Rerank-RRF-Hybrid (0.1550)

✅ Output: /kaggle/working/evaluation_runs/ablation3_reranker
